# WiDS Global Datathon 2026 | Hybrid Survival Stack v2

In [1]:
import os, sys, json, math, warnings, subprocess
warnings.filterwarnings("ignore")

RUN_MODE = "full"   # "full" or "fast"
DO_OOF = True
AUTO_INSTALL_SKSURV = True
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUTPUT_PATH = os.path.join(WORK_DIR, "submission.csv")

def find_data_dir():
    candidates = [
        "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/widsworldwide-globaldathon26",
        "/kaggle/input/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/wiDSworldwide_globaldathon26",
        ".",
        "/mnt/data",
    ]
    for path in candidates:
        if all(os.path.exists(os.path.join(path, f)) for f in ["train.csv", "test.csv", "sample_submission.csv"]):
            return path
    for root, dirs, files in os.walk("/kaggle/input"):
        files = set(files)
        if {"train.csv", "test.csv", "sample_submission.csv"}.issubset(files):
            return root
    raise FileNotFoundError("Could not find competition data directory.")

DATA_DIR = find_data_dir()

def ensure_pkg(pkg, import_name=None, allow_install=True):
    name = import_name or pkg
    try:
        __import__(name)
        return True
    except Exception:
        if not allow_install:
            return False
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
            __import__(name)
            return True
        except Exception as e:
            print(f"[WARN] package unavailable: {pkg} -> {e}")
            return False

HAS_SKSURV = ensure_pkg("scikit-survival", "sksurv", AUTO_INSTALL_SKSURV)
ensure_pkg("lightgbm", "lightgbm", True)
ensure_pkg("catboost", "catboost", True)

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb
from catboost import CatBoostClassifier

if HAS_SKSURV:
    from sksurv.util import Surv
    from sksurv.ensemble import GradientBoostingSurvivalAnalysis, RandomSurvivalForest
    from sksurv.linear_model import CoxPHSurvivalAnalysis

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

print("DATA_DIR =", DATA_DIR)
print("TRAIN =", train_df.shape, "TEST =", test_df.shape, "HAS_SKSURV =", HAS_SKSURV)

def create_features(df, fit_df=None):
    ref = fit_df if fit_df is not None else df
    r = df.copy()

    dist = r["dist_min_ci_0_5h"].clip(lower=1)
    speed = r["closing_speed_m_per_h"]
    perims = r["num_perimeters_0_5h"]
    area_first = r["area_first_ha"]
    area_growth = r["area_growth_rate_ha_per_h"]
    radial = r["radial_growth_rate_m_per_h"].clip(lower=0)
    align = r["alignment_abs"]
    fire_radius = np.sqrt(np.maximum(area_first, 0) * 10000.0 / np.pi)
    closing_pos = speed.clip(lower=0)
    eff_close = closing_pos + radial

    r["dist_km"] = dist / 1000.0
    r["log_distance"] = np.log1p(dist)
    r["inv_distance"] = 1.0 / (dist / 1000.0 + 0.1)
    r["inv_distance_sq"] = r["inv_distance"] ** 2
    r["sqrt_distance"] = np.sqrt(dist)
    r["dist_km_sq"] = r["dist_km"] ** 2
    r["dist_km_cb"] = r["dist_km"] ** 3

    r["fire_radius_km"] = fire_radius / 1000.0
    r["radius_to_dist"] = fire_radius / dist
    r["area_to_dist_ratio"] = area_first / (dist / 1000.0 + 0.1)
    r["log_area_dist_ratio"] = np.log1p(area_first) - np.log1p(dist)

    r["has_movement"] = (perims > 1).astype(float)
    r["effective_closing_speed"] = eff_close
    r["eta_hours"] = np.where(closing_pos > 0.01, dist / closing_pos, 9999.0).clip(max=9999.0)
    r["eta_effective"] = np.where(eff_close > 0.01, dist / eff_close, 9999.0).clip(max=9999.0)
    r["log_eta"] = np.log1p(r["eta_hours"])
    r["log_eta_effective"] = np.log1p(r["eta_effective"])

    r["threat_score"] = align * speed / np.log1p(dist)
    r["threat_score_eff"] = align * eff_close / np.log1p(dist)
    r["threat_score_sq"] = r["threat_score_eff"] ** 2
    r["fire_urgency"] = perims * speed
    r["growth_intensity"] = area_growth * perims
    r["speed_alignment"] = speed * align
    r["effective_alignment"] = eff_close * align

    r["projected_advance_pos"] = np.maximum(r["projected_advance_m"], 0)
    r["closing_distance_pos"] = np.maximum(-r["dist_change_ci_0_5h"], 0)
    r["slope_toward"] = np.maximum(-r["dist_slope_ci_0_5h"], 0)
    r["accel_toward"] = np.maximum(-r["dist_accel_m_per_h2"], 0)

    r["zone_3km"] = (dist < 3000).astype(float)
    r["zone_near"] = (dist < 5000).astype(float)
    r["zone_warning"] = ((dist >= 5000) & (dist < 10000)).astype(float)
    r["zone_far"] = (dist >= 10000).astype(float)
    r["zone_20km"] = (dist < 20000).astype(float)

    r["is_summer"] = r["event_start_month"].isin([6, 7, 8]).astype(float)
    r["is_afternoon"] = ((r["event_start_hour"] >= 12) & (r["event_start_hour"] < 20)).astype(float)
    r["is_night"] = ((r["event_start_hour"] <= 6) | (r["event_start_hour"] >= 21)).astype(float)

    r["hour_sin"] = np.sin(2 * np.pi * r["event_start_hour"] / 24.0)
    r["hour_cos"] = np.cos(2 * np.pi * r["event_start_hour"] / 24.0)
    r["month_sin"] = np.sin(2 * np.pi * (r["event_start_month"] - 1) / 12.0)
    r["month_cos"] = np.cos(2 * np.pi * (r["event_start_month"] - 1) / 12.0)
    r["dow_sin"] = np.sin(2 * np.pi * r["event_start_dayofweek"] / 7.0)
    r["dow_cos"] = np.cos(2 * np.pi * r["event_start_dayofweek"] / 7.0)

    def pct_rank(values, ref_values):
        ref_values = np.asarray(ref_values, dtype=float)
        if ref_values.size == 0:
            return np.zeros(len(values), dtype=float)
        ref_sorted = np.sort(ref_values)
        return np.searchsorted(ref_sorted, np.asarray(values, dtype=float), side="right") / ref_sorted.size

    ref_dist = ref["dist_min_ci_0_5h"].clip(lower=1).values
    ref_eff = (ref["closing_speed_m_per_h"].clip(lower=0).values +
               ref["radial_growth_rate_m_per_h"].clip(lower=0).values)
    ref_threat = (
        ref["alignment_abs"].values * ref_eff /
        np.log1p(ref["dist_min_ci_0_5h"].clip(lower=1).values)
    )

    r["dist_rank_global"] = pct_rank(dist.values, ref_dist)
    r["eff_rank_global"] = pct_rank(eff_close.values, ref_eff)
    r["threat_rank_global"] = pct_rank(r["threat_score_eff"].values, ref_threat)

    ref_near = ref["dist_min_ci_0_5h"].clip(lower=1).values < 5000
    cur_near = dist.values < 5000
    near_speed_rank = np.zeros(len(r), dtype=float)
    far_threat_rank = np.zeros(len(r), dtype=float)
    near_ref_eff = ref_eff[ref_near]
    far_ref_threat = ref_threat[~ref_near]
    if cur_near.any():
        near_speed_rank[cur_near] = pct_rank(eff_close.values[cur_near], near_ref_eff)
    if (~cur_near).any():
        far_threat_rank[~cur_near] = pct_rank(r["threat_score_eff"].values[~cur_near], far_ref_threat)
    r["near_speed_rank"] = near_speed_rank
    r["far_threat_rank"] = far_threat_rank

    drop_cols = [
        "relative_growth_0_5h",
        "projected_advance_m",
        "centroid_displacement_m",
        "centroid_speed_m_per_h",
        "closing_speed_abs_m_per_h",
        "area_growth_abs_0_5h",
    ]
    r = r.drop(columns=[c for c in drop_cols if c in r.columns])
    r = r.replace([np.inf, -np.inf], np.nan).fillna(0)
    return r

train_proc = create_features(train_df, fit_df=train_df)
test_proc = create_features(test_df, fit_df=train_df)

time_values = train_df["time_to_hit_hours"].values
event_values = train_df["event"].astype(int).values
dist_train = train_df["dist_min_ci_0_5h"].values
dist_test = test_df["dist_min_ci_0_5h"].values

def compute_c_index(time, event, risk):
    n = len(time)
    conc = 0.0
    comp = 0.0
    for i in range(n):
        if event[i] != 1:
            continue
        for j in range(n):
            if i == j or time[i] >= time[j]:
                continue
            comp += 1
            if risk[i] > risk[j]:
                conc += 1
            elif risk[i] == risk[j]:
                conc += 0.5
    return conc / comp if comp else 0.5

def compute_brier(time, event, prob, horizon):
    valid = ~((event == 0) & (time < horizon))
    if valid.sum() == 0:
        return 0.25
    y_true = ((event == 1) & (time <= horizon)).astype(float)[valid]
    prob = np.clip(prob[valid], 0, 1)
    return float(np.mean((prob - y_true) ** 2))

def compute_hybrid_score(time, event, p24, p48, p72):
    risk = 0.3 * p24 + 0.4 * p48 + 0.3 * p72
    c_idx = compute_c_index(time, event, risk)
    b24 = compute_brier(time, event, p24, 24)
    b48 = compute_brier(time, event, p48, 48)
    b72 = compute_brier(time, event, p72, 72)
    wb = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    return 0.3 * c_idx + 0.7 * (1.0 - wb), c_idx, wb, b24, b48, b72

def enforce_monotonicity(preds):
    out = np.clip(preds, 0, 1)
    for i in range(1, out.shape[1]):
        out[:, i] = np.maximum(out[:, i], out[:, i - 1])
    return out

def make_binary_target(time_vals, event_vals, horizon):
    unknown = (event_vals == 0) & (time_vals < horizon)
    y = ((event_vals == 1) & (time_vals <= horizon)).astype(int)
    return y, ~unknown

def compute_ipcw_weights(times, events, horizon):
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    unique_t = np.sort(np.unique(times))
    surv = np.ones(len(unique_t), dtype=float)
    for i, t in enumerate(unique_t):
        at_risk = np.sum(times >= t)
        cens = np.sum((times == t) & (events == 0))
        if at_risk > 0:
            surv[i] = 1.0 - cens / at_risk
        if i > 0:
            surv[i] *= surv[i - 1]
    def G(t):
        idx = np.searchsorted(unique_t, t, side="right") - 1
        if idx >= 0:
            return max(float(surv[idx]), 0.01)
        return 1.0
    weights = np.ones(len(times), dtype=float)
    for i in range(len(times)):
        if events[i] == 1 and times[i] <= horizon:
            weights[i] = 1.0 / G(times[i])
        elif times[i] >= horizon:
            weights[i] = 1.0 / G(horizon)
        else:
            weights[i] = 0.0
    return weights

def fit_with_optional_weight(model, X, y, sample_weight=None):
    if sample_weight is None:
        model.fit(X, y)
        return model
    if isinstance(model, Pipeline):
        last_name = model.steps[-1][0]
        try:
            model.fit(X, y, **{f"{last_name}__sample_weight": sample_weight})
            return model
        except Exception:
            model.fit(X, y)
            return model
    try:
        model.fit(X, y, sample_weight=sample_weight)
        return model
    except Exception:
        model.fit(X, y)
        return model

def safe_predict_proba(model, X):
    p = model.predict_proba(X)
    if isinstance(p, list):
        p = np.asarray(p)
    return p[:, 1] if p.ndim == 2 else p

def soft_gate(dist_m, center, width):
    z = np.clip((dist_m - center) / max(width, 1.0), -60, 60)
    return 1.0 / (1.0 + np.exp(z))

def stabilize(pred, anchor, alpha):
    return np.clip(alpha * pred + (1.0 - alpha) * anchor, 0, 1)

def repeated_isotonic(train_signal, test_signal, y, mask, seeds, n_splits=5):
    train_signal = np.asarray(train_signal, dtype=float)
    test_signal = np.asarray(test_signal, dtype=float)
    valid_idx = np.where(mask)[0]
    yv = y[valid_idx]
    oof = np.zeros(len(train_signal), dtype=float)
    cnt = np.zeros(len(train_signal), dtype=float)
    full_fill = np.zeros(len(train_signal), dtype=float)
    test_pred = np.zeros(len(test_signal), dtype=float)

    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_sub, va_sub in cv.split(valid_idx, yv):
            tr_idx = valid_idx[tr_sub]
            va_idx = valid_idx[va_sub]
            ir = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
            ir.fit(train_signal[tr_idx], y[tr_idx])
            oof[va_idx] += ir.predict(train_signal[va_idx])
            cnt[va_idx] += 1.0

        ir_full = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
        ir_full.fit(train_signal[valid_idx], yv)
        full_fill += ir_full.predict(train_signal)
        test_pred += ir_full.predict(test_signal)

    full_fill /= len(seeds)
    test_pred /= len(seeds)
    nz = cnt > 0
    oof[nz] /= cnt[nz]
    oof[~nz] = full_fill[~nz]
    return np.clip(oof, 0, 1), np.clip(test_pred, 0, 1)

def repeated_binary_model(X_train, X_test, y, mask, build_model, seeds, n_splits=5, horizon=None, use_ipcw=False):
    n_train = len(X_train)
    n_test = len(X_test)
    valid_idx = np.where(mask)[0]
    Xv = X_train.iloc[valid_idx]
    yv = y[valid_idx]
    oof = np.zeros(n_train, dtype=float)
    cnt = np.zeros(n_train, dtype=float)
    full_fill = np.zeros(n_train, dtype=float)
    test_pred = np.zeros(n_test, dtype=float)

    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_sub, va_sub in cv.split(Xv, yv):
            tr_idx = valid_idx[tr_sub]
            va_idx = valid_idx[va_sub]
            model = build_model(seed)
            sw = None
            if use_ipcw and horizon is not None:
                sw = compute_ipcw_weights(time_values[tr_idx], event_values[tr_idx], horizon)
            fit_with_optional_weight(model, X_train.iloc[tr_idx], y[tr_idx], sw)
            oof[va_idx] += safe_predict_proba(model, X_train.iloc[va_idx])
            cnt[va_idx] += 1.0

        model_full = build_model(seed)
        sw_full = None
        if use_ipcw and horizon is not None:
            sw_full = compute_ipcw_weights(time_values[valid_idx], event_values[valid_idx], horizon)
        fit_with_optional_weight(model_full, Xv, yv, sw_full)
        full_fill += safe_predict_proba(model_full, X_train)
        test_pred += safe_predict_proba(model_full, X_test)

    full_fill /= len(seeds)
    test_pred /= len(seeds)
    nz = cnt > 0
    oof[nz] /= cnt[nz]
    oof[~nz] = full_fill[~nz]
    return np.clip(oof, 0, 1), np.clip(test_pred, 0, 1)

if HAS_SKSURV:
    HORIZONS = [12, 24, 48, 72]

    def get_surv_predictions(model, X):
        surv_fns = model.predict_survival_function(X)
        preds = np.empty((len(surv_fns), len(HORIZONS)), dtype=float)
        for i, fn in enumerate(surv_fns):
            try:
                t_min, t_max = fn.domain
            except Exception:
                t_min, t_max = fn.x[0], fn.x[-1]
            grid = np.clip(HORIZONS, t_min, t_max)
            preds[i, :] = 1.0 - fn(grid)
        return np.clip(preds, 0, 1)

    y_surv = Surv.from_arrays(
        event=train_df["event"].astype(bool).values,
        time=train_df["time_to_hit_hours"].values
    )

    def repeated_survival_model(X_train, X_test, build_model, seeds, n_splits=5):
        n_train = len(X_train)
        n_test = len(X_test)
        oof = np.zeros((n_train, 4), dtype=float)
        cnt = np.zeros(n_train, dtype=float)
        full_fill = np.zeros((n_train, 4), dtype=float)
        test_pred = np.zeros((n_test, 4), dtype=float)

        for seed in seeds:
            cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
            seed_test = np.zeros((n_test, 4), dtype=float)

            for tr_idx, va_idx in cv.split(X_train, event_values):
                model = build_model(seed)
                try:
                    model.fit(X_train.iloc[tr_idx], y_surv[tr_idx])
                    oof[va_idx] += get_surv_predictions(model, X_train.iloc[va_idx])
                    cnt[va_idx] += 1.0
                    seed_test += get_surv_predictions(model, X_test) / n_splits
                except Exception:
                    oof[va_idx] += 0.5
                    cnt[va_idx] += 1.0
                    seed_test += 0.5 / n_splits

            test_pred += seed_test / len(seeds)

            model_full = build_model(seed)
            try:
                model_full.fit(X_train, y_surv)
                full_fill += get_surv_predictions(model_full, X_train)
            except Exception:
                full_fill += 0.5

        full_fill /= len(seeds)
        nz = cnt > 0
        oof[nz] /= cnt[nz, None]
        oof[~nz] = full_fill[~nz]
        return np.clip(oof, 0, 1), np.clip(test_pred, 0, 1)

FAST_SEEDS = (123, 456, 789)
BASE_SEEDS = (123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033, 279, 239, 70, 77, 31, 2024)
GBSA_SEEDS_FULL = BASE_SEEDS + (2077, 3077, 4640, 841, 7755, 8525, 2701, 8817)
COX_SEEDS_FULL = BASE_SEEDS[:12]
RSF_SEEDS_FULL = BASE_SEEDS[:10]
TREE_SEEDS_FULL = BASE_SEEDS + (2077, 3077, 123456, 654321)
CAT_SEEDS_FULL = BASE_SEEDS[:8]

def choose_seeds(full_seeds):
    return FAST_SEEDS if RUN_MODE == "fast" else tuple(full_seeds)

RAW_FEATURES = [c for c in train_df.columns if c not in ["event_id", "event", "time_to_hit_hours"]]

COX_FEATURES = [
    "dist_km", "log_distance", "inv_distance",
    "closing_speed_m_per_h", "radial_growth_rate_m_per_h",
    "alignment_abs", "threat_score_eff", "log_eta_effective", "eta_effective",
    "area_to_dist_ratio", "fire_radius_km",
    "num_perimeters_0_5h", "has_movement",
    "near_speed_rank", "far_threat_rank",
    "low_temporal_resolution_0_5h", "dt_first_last_0_5h",
    "is_summer", "is_afternoon", "hour_sin", "hour_cos",
    "zone_near", "zone_far",
]
COX_FEATURES = [c for c in COX_FEATURES if c in train_proc.columns]

NEAR_LGB_FEATURES = [
    "closing_speed_m_per_h", "radial_growth_rate_m_per_h",
    "alignment_abs", "num_perimeters_0_5h", "area_growth_rate_ha_per_h",
    "eta_effective", "log_eta_effective", "dist_km", "threat_score_eff",
    "near_speed_rank", "event_start_hour", "is_afternoon", "fire_urgency",
    "area_first_ha", "fire_radius_km", "low_temporal_resolution_0_5h",
    "dt_first_last_0_5h",
]
NEAR_LGB_FEATURES = [c for c in NEAR_LGB_FEATURES if c in train_proc.columns]

FAR_LGB_FEATURES = [
    "dist_km", "log_distance", "inv_distance",
    "closing_speed_m_per_h", "alignment_abs",
    "threat_score_eff", "log_eta_effective", "eta_effective",
    "area_to_dist_ratio", "num_perimeters_0_5h",
    "far_threat_rank", "is_summer", "zone_far",
    "low_temporal_resolution_0_5h", "dt_first_last_0_5h",
    "log1p_growth", "dist_fit_r2_0_5h",
]
FAR_LGB_FEATURES = [c for c in FAR_LGB_FEATURES if c in train_proc.columns]

GLOBAL_TREE_FEATURES = sorted(set(
    NEAR_LGB_FEATURES + FAR_LGB_FEATURES + [
        "inv_distance_sq", "sqrt_distance", "threat_score", "growth_intensity",
        "speed_alignment", "effective_alignment", "zone_warning",
        "hour_sin", "hour_cos", "month_sin", "month_cos", "dow_sin", "dow_cos",
        "projected_advance_pos", "closing_distance_pos", "slope_toward",
        "log_area_dist_ratio", "fire_radius_km", "log1p_area_first",
        "dist_rank_global", "eff_rank_global", "threat_rank_global",
    ]
))
GLOBAL_TREE_FEATURES = [c for c in GLOBAL_TREE_FEATURES if c in train_proc.columns]

LINEAR_FEATURES = [
    "dist_km", "log_distance", "inv_distance",
    "closing_speed_m_per_h", "radial_growth_rate_m_per_h",
    "alignment_abs", "threat_score_eff", "log_eta_effective", "eta_effective",
    "area_to_dist_ratio", "fire_radius_km",
    "num_perimeters_0_5h", "has_movement",
    "near_speed_rank", "far_threat_rank",
    "low_temporal_resolution_0_5h", "dt_first_last_0_5h",
    "is_summer", "is_afternoon", "hour_sin", "hour_cos",
    "zone_near", "zone_far",
]
LINEAR_FEATURES = [c for c in LINEAR_FEATURES if c in train_proc.columns]

X_raw_train = train_df[RAW_FEATURES].copy()
X_raw_test = test_df[RAW_FEATURES].copy()

if HAS_SKSURV:
    coxt = train_proc[COX_FEATURES].copy()
    non_const = coxt.std(axis=0) > 0
    COX_FEATURES = list(coxt.columns[non_const.values])
    scaler_cox = StandardScaler()
    X_cox_train = pd.DataFrame(
        scaler_cox.fit_transform(train_proc[COX_FEATURES]),
        columns=COX_FEATURES, index=train_proc.index
    )
    X_cox_test = pd.DataFrame(
        scaler_cox.transform(test_proc[COX_FEATURES]),
        columns=COX_FEATURES, index=test_proc.index
    )

X_near_lgb_train = train_proc[NEAR_LGB_FEATURES].copy()
X_near_lgb_test = test_proc[NEAR_LGB_FEATURES].copy()
X_far_lgb_train = train_proc[FAR_LGB_FEATURES].copy()
X_far_lgb_test = test_proc[FAR_LGB_FEATURES].copy()
X_global_train = train_proc[GLOBAL_TREE_FEATURES].copy()
X_global_test = test_proc[GLOBAL_TREE_FEATURES].copy()
X_linear_train = train_proc[LINEAR_FEATURES].copy()
X_linear_test = test_proc[LINEAR_FEATURES].copy()

def make_lgb_builder(params):
    def _build(seed):
        p = dict(params)
        p.update(dict(objective="binary", random_state=seed, n_jobs=-1, verbose=-1))
        return lgb.LGBMClassifier(**p)
    return _build

def make_cat_builder(params):
    def _build(seed):
        p = dict(params)
        p.update(dict(
            loss_function="Logloss",
            eval_metric="Logloss",
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        ))
        return CatBoostClassifier(**p)
    return _build

def make_logit_builder(C):
    def _build(seed):
        return Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(C=C, penalty="l2", solver="lbfgs", max_iter=4000)),
        ])
    return _build

gbsa_configs = [
    {"learning_rate": 0.01,  "subsample": 0.70, "max_depth": 3, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 1200},
    {"learning_rate": 0.01,  "subsample": 0.85, "max_depth": 3, "min_samples_leaf": 15, "min_samples_split": 3, "n_estimators": 1200},
    {"learning_rate": 0.005, "subsample": 0.85, "max_depth": 3, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 2000},
    {"learning_rate": 0.01,  "subsample": 0.85, "max_depth": 3, "min_samples_leaf": 20, "min_samples_split": 3, "n_estimators": 1400},
    {"learning_rate": 0.008, "subsample": 0.75, "max_depth": 2, "min_samples_leaf": 15, "min_samples_split": 4, "n_estimators": 1500},
    {"learning_rate": 0.015, "subsample": 0.70, "max_depth": 3, "min_samples_leaf": 10, "min_samples_split": 3, "n_estimators": 1000},
    {"learning_rate": 0.005, "subsample": 0.90, "max_depth": 3, "min_samples_leaf": 18, "min_samples_split": 5, "n_estimators": 2500},
    {"learning_rate": 0.01,  "subsample": 0.80, "max_depth": 4, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 1200},
]
cox_alphas = [0.001, 0.01, 0.05, 0.10, 0.50, 1.00, 2.00]
rsf_configs = [
    {"n_estimators": 200, "min_samples_leaf": 12, "max_features": "sqrt", "max_depth": None},
    {"n_estimators": 200, "min_samples_leaf": 18, "max_features": "sqrt", "max_depth": None},
    {"n_estimators": 200, "min_samples_leaf": 12, "max_features": 0.5,    "max_depth": 5},
]

if RUN_MODE == "fast":
    gbsa_configs = gbsa_configs[:3]
    cox_alphas = [0.01, 0.10, 1.00]
    rsf_configs = rsf_configs[:1]

global_lgb_cfgs = {
    12: {"max_depth": 2, "learning_rate": 0.040, "n_estimators": 220, "subsample": 0.80, "colsample_bytree": 0.80,
         "min_child_samples": 4, "reg_alpha": 0.5, "reg_lambda": 1.5, "num_leaves": 4},
    24: {"max_depth": 2, "learning_rate": 0.035, "n_estimators": 260, "subsample": 0.80, "colsample_bytree": 0.80,
         "min_child_samples": 6, "reg_alpha": 0.8, "reg_lambda": 2.5, "num_leaves": 4},
    48: {"max_depth": 3, "learning_rate": 0.030, "n_estimators": 260, "subsample": 0.85, "colsample_bytree": 0.85,
         "min_child_samples": 8, "reg_alpha": 1.0, "reg_lambda": 3.0, "num_leaves": 6},
}

global_cat_cfgs = {
    12: {"iterations": 300, "learning_rate": 0.030, "depth": 4, "l2_leaf_reg": 10.0,
         "bootstrap_type": "Bernoulli", "subsample": 0.85, "random_strength": 0.8},
    24: {"iterations": 320, "learning_rate": 0.030, "depth": 4, "l2_leaf_reg": 12.0,
         "bootstrap_type": "Bernoulli", "subsample": 0.85, "random_strength": 0.8},
    48: {"iterations": 340, "learning_rate": 0.028, "depth": 4, "l2_leaf_reg": 14.0,
         "bootstrap_type": "Bernoulli", "subsample": 0.85, "random_strength": 0.8},
}

near_lgb_cfgs = {
    12: {"max_depth": 2, "learning_rate": 0.040, "n_estimators": 220, "subsample": 0.80, "colsample_bytree": 0.80,
         "min_child_samples": 4, "reg_alpha": 0.5, "reg_lambda": 1.5, "num_leaves": 4},
    24: {"max_depth": 2, "learning_rate": 0.040, "n_estimators": 200, "subsample": 0.80, "colsample_bytree": 0.80,
         "min_child_samples": 4, "reg_alpha": 0.5, "reg_lambda": 1.5, "num_leaves": 4},
    48: {"max_depth": 2, "learning_rate": 0.050, "n_estimators": 150, "subsample": 0.80, "colsample_bytree": 0.80,
         "min_child_samples": 3, "reg_alpha": 0.3, "reg_lambda": 1.0, "num_leaves": 4},
}

far_lgb_cfgs = {
    24: {"max_depth": 2, "learning_rate": 0.030, "n_estimators": 200, "subsample": 0.70, "colsample_bytree": 0.70,
         "min_child_samples": 8, "reg_alpha": 1.0, "reg_lambda": 3.0, "num_leaves": 4},
    48: {"max_depth": 2, "learning_rate": 0.050, "n_estimators": 150, "subsample": 0.80, "colsample_bytree": 0.80,
         "min_child_samples": 6, "reg_alpha": 0.5, "reg_lambda": 2.0, "num_leaves": 4},
}

linear_cs = {12: 0.60, 24: 0.80, 48: 1.00}

y_map = {}
mask_map = {}
for h in [12, 24, 48]:
    y_map[h], mask_map[h] = make_binary_target(time_values, event_values, h)

ANCHOR_SEEDS = choose_seeds(BASE_SEEDS[:8])
TREE_SEEDS = choose_seeds(TREE_SEEDS_FULL)
CAT_SEEDS = choose_seeds(CAT_SEEDS_FULL)
LINEAR_SEEDS = choose_seeds(BASE_SEEDS[:8])

anchor_signal_train = 1.0 / (train_proc["dist_km"].values + 0.25)
anchor_signal_test = 1.0 / (test_proc["dist_km"].values + 0.25)

anchor_oof, anchor_test = {}, {}
for h in [12, 24, 48]:
    anchor_oof[h], anchor_test[h] = repeated_isotonic(
        anchor_signal_train, anchor_signal_test, y_map[h], mask_map[h], ANCHOR_SEEDS, n_splits=5
    )
    print(f"anchor@{h}  Brier = {compute_brier(time_values, event_values, anchor_oof[h], h):.6f}")

global_lgb_oof, global_lgb_test = {}, {}
global_cat_oof, global_cat_test = {}, {}
linear_oof, linear_test = {}, {}

for h in [12, 24, 48]:
    global_lgb_oof[h], global_lgb_test[h] = repeated_binary_model(
        X_global_train, X_global_test, y_map[h], mask_map[h],
        make_lgb_builder(global_lgb_cfgs[h]), TREE_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48])
    )
    global_lgb_oof[h] = stabilize(global_lgb_oof[h], anchor_oof[h], 0.92)
    global_lgb_test[h] = stabilize(global_lgb_test[h], anchor_test[h], 0.92)
    print(f"global_lgb@{h}  Brier = {compute_brier(time_values, event_values, global_lgb_oof[h], h):.6f}")

    global_cat_oof[h], global_cat_test[h] = repeated_binary_model(
        X_global_train, X_global_test, y_map[h], mask_map[h],
        make_cat_builder(global_cat_cfgs[h]), CAT_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48])
    )
    global_cat_oof[h] = stabilize(global_cat_oof[h], anchor_oof[h], 0.90)
    global_cat_test[h] = stabilize(global_cat_test[h], anchor_test[h], 0.90)
    print(f"global_cat@{h}  Brier = {compute_brier(time_values, event_values, global_cat_oof[h], h):.6f}")

    linear_oof[h], linear_test[h] = repeated_binary_model(
        X_linear_train, X_linear_test, y_map[h], mask_map[h],
        make_logit_builder(linear_cs[h]), LINEAR_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48])
    )
    linear_oof[h] = stabilize(linear_oof[h], anchor_oof[h], 0.88)
    linear_test[h] = stabilize(linear_test[h], anchor_test[h], 0.88)
    print(f"linear@{h}      Brier = {compute_brier(time_values, event_values, linear_oof[h], h):.6f}")

near_lgb_oof, near_lgb_test = {}, {}
for h in [12, 24, 48]:
    near_lgb_oof[h], near_lgb_test[h] = repeated_binary_model(
        X_near_lgb_train, X_near_lgb_test, y_map[h], mask_map[h],
        make_lgb_builder(near_lgb_cfgs[h]), TREE_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48])
    )
    near_lgb_oof[h] = stabilize(near_lgb_oof[h], anchor_oof[h], 0.94)
    near_lgb_test[h] = stabilize(near_lgb_test[h], anchor_test[h], 0.94)
    print(f"near_lgb@{h}    Brier = {compute_brier(time_values, event_values, near_lgb_oof[h], h):.6f}")

far_lgb_oof, far_lgb_test = {}, {}
for h in [24, 48]:
    far_lgb_oof[h], far_lgb_test[h] = repeated_binary_model(
        X_far_lgb_train, X_far_lgb_test, y_map[h], mask_map[h],
        make_lgb_builder(far_lgb_cfgs[h]), TREE_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48])
    )
    far_lgb_oof[h] = stabilize(far_lgb_oof[h], anchor_oof[h], 0.94)
    far_lgb_test[h] = stabilize(far_lgb_test[h], anchor_test[h], 0.94)
    print(f"far_lgb@{h}     Brier = {compute_brier(time_values, event_values, far_lgb_oof[h], h):.6f}")

timing_signal_train = 1.0 / (np.asarray(train_proc["eta_effective"], dtype=float) / 24.0 + 1.0)
timing_signal_test = 1.0 / (np.asarray(test_proc["eta_effective"], dtype=float) / 24.0 + 1.0)
timing_oof, timing_test = {}, {}
for h in [12, 24, 48]:
    timing_mask = mask_map[h] & ((dist_train < 8000) | (y_map[h] == 1))
    timing_oof[h], timing_test[h] = repeated_isotonic(
        timing_signal_train, timing_signal_test, y_map[h], timing_mask, ANCHOR_SEEDS, n_splits=5
    )
    timing_oof[h] = stabilize(timing_oof[h], anchor_oof[h], 0.90)
    timing_test[h] = stabilize(timing_test[h], anchor_test[h], 0.90)
    print(f"timing@{h}      Brier = {compute_brier(time_values, event_values, timing_oof[h], h):.6f}")

base_oof = {
    "anchor": anchor_oof,
    "glob_lgb": global_lgb_oof,
    "glob_cat": global_cat_oof,
    "lin": linear_oof,
    "timing": timing_oof,
}
base_test = {
    "anchor": anchor_test,
    "glob_lgb": global_lgb_test,
    "glob_cat": global_cat_test,
    "lin": linear_test,
    "timing": timing_test,
}

if HAS_SKSURV:
    GBSA_SEEDS = choose_seeds(GBSA_SEEDS_FULL)
    COX_SEEDS = choose_seeds(COX_SEEDS_FULL)
    RSF_SEEDS = choose_seeds(RSF_SEEDS_FULL)

    gbsa_oof = np.zeros((len(train_df), 4), dtype=float)
    gbsa_test = np.zeros((len(test_df), 4), dtype=float)
    for cfg in gbsa_configs:
        oof_cfg, test_cfg = repeated_survival_model(
            X_raw_train, X_raw_test,
            lambda seed, cfg=cfg: GradientBoostingSurvivalAnalysis(random_state=seed, **cfg),
            GBSA_SEEDS, n_splits=5
        )
        gbsa_oof += oof_cfg / len(gbsa_configs)
        gbsa_test += test_cfg / len(gbsa_configs)

    gbsa_oof[:, 1] = np.clip(gbsa_oof[:, 1] ** 1.08, 0, 1)
    gbsa_test[:, 1] = np.clip(gbsa_test[:, 1] ** 1.08, 0, 1)
    for i, h in enumerate([12, 24, 48]):
        gbsa_oof[:, i] = stabilize(gbsa_oof[:, i], anchor_oof[h], 0.98 if h != 24 else 0.97)
        gbsa_test[:, i] = stabilize(gbsa_test[:, i], anchor_test[h], 0.98 if h != 24 else 0.97)
        print(f"gbsa@{h}        Brier = {compute_brier(time_values, event_values, gbsa_oof[:, i], h):.6f}")

    cox_oof = np.zeros((len(train_df), 4), dtype=float)
    cox_test = np.zeros((len(test_df), 4), dtype=float)
    for alpha in cox_alphas:
        oof_alpha, test_alpha = repeated_survival_model(
            X_cox_train, X_cox_test,
            lambda seed, a=alpha: CoxPHSurvivalAnalysis(alpha=a),
            COX_SEEDS, n_splits=5
        )
        cox_oof += oof_alpha / len(cox_alphas)
        cox_test += test_alpha / len(cox_alphas)
    for i, h in enumerate([12, 24, 48]):
        cox_oof[:, i] = stabilize(cox_oof[:, i], anchor_oof[h], 0.97)
        cox_test[:, i] = stabilize(cox_test[:, i], anchor_test[h], 0.97)
        print(f"cox@{h}         Brier = {compute_brier(time_values, event_values, cox_oof[:, i], h):.6f}")

    rsf_oof = np.zeros((len(train_df), 4), dtype=float)
    rsf_test = np.zeros((len(test_df), 4), dtype=float)
    for cfg in rsf_configs:
        oof_cfg, test_cfg = repeated_survival_model(
            X_raw_train, X_raw_test,
            lambda seed, cfg=cfg: RandomSurvivalForest(random_state=seed, n_jobs=-1, **cfg),
            RSF_SEEDS, n_splits=5
        )
        rsf_oof += oof_cfg / len(rsf_configs)
        rsf_test += test_cfg / len(rsf_configs)
    for i, h in enumerate([12, 24, 48]):
        rsf_oof[:, i] = stabilize(rsf_oof[:, i], anchor_oof[h], 0.90)
        rsf_test[:, i] = stabilize(rsf_test[:, i], anchor_test[h], 0.90)
        print(f"rsf@{h}         Brier = {compute_brier(time_values, event_values, rsf_oof[:, i], h):.6f}")

    base_oof["gbsa"] = {12: gbsa_oof[:, 0], 24: gbsa_oof[:, 1], 48: gbsa_oof[:, 2]}
    base_oof["cox"] = {12: cox_oof[:, 0], 24: cox_oof[:, 1], 48: cox_oof[:, 2]}
    base_oof["rsf"] = {12: rsf_oof[:, 0], 24: rsf_oof[:, 1], 48: rsf_oof[:, 2]}

    base_test["gbsa"] = {12: gbsa_test[:, 0], 24: gbsa_test[:, 1], 48: gbsa_test[:, 2]}
    base_test["cox"] = {12: cox_test[:, 0], 24: cox_test[:, 1], 48: cox_test[:, 2]}
    base_test["rsf"] = {12: rsf_test[:, 0], 24: rsf_test[:, 1], 48: rsf_test[:, 2]}
else:
    print("[WARN] scikit-survival unavailable: using classifier stack + isotonic anchor only.")

near_lgb_store = {"oof": near_lgb_oof, "test": near_lgb_test}
far_lgb_store = {"oof": far_lgb_oof, "test": far_lgb_test}

PRIORS = {
    (12, "near"): {"gbsa": 0.60, "cox": 0.10, "rsf": 0.02, "zone_lgb": 0.10, "timing": 0.06, "glob_lgb": 0.05, "glob_cat": 0.03, "lin": 0.02, "anchor": 0.02},
    (12, "far"):  {"gbsa": 0.78, "glob_lgb": 0.08, "glob_cat": 0.05, "lin": 0.04, "anchor": 0.05},
    (24, "near"): {"gbsa": 0.66, "cox": 0.12, "rsf": 0.02, "zone_lgb": 0.05, "timing": 0.05, "glob_lgb": 0.04, "glob_cat": 0.03, "lin": 0.02, "anchor": 0.01},
    (24, "far"):  {"gbsa": 0.52, "cox": 0.20, "rsf": 0.05, "zone_lgb": 0.10, "glob_lgb": 0.06, "glob_cat": 0.04, "lin": 0.02, "anchor": 0.01},
    (48, "near"): {"gbsa": 0.54, "cox": 0.14, "rsf": 0.03, "zone_lgb": 0.09, "timing": 0.06, "glob_lgb": 0.06, "glob_cat": 0.04, "lin": 0.02, "anchor": 0.02},
    (48, "far"):  {"gbsa": 0.26, "cox": 0.18, "rsf": 0.06, "zone_lgb": 0.22, "glob_lgb": 0.14, "glob_cat": 0.08, "lin": 0.03, "anchor": 0.03},
}
BLEND_CFG = {
    (12, "near"): {"shrink": 0.25, "reg": 0.12},
    (12, "far"):  {"shrink": 0.35, "reg": 0.10},
    (24, "near"): {"shrink": 0.30, "reg": 0.10},
    (24, "far"):  {"shrink": 0.45, "reg": 0.08},
    (48, "near"): {"shrink": 0.35, "reg": 0.08},
    (48, "far"):  {"shrink": 0.50, "reg": 0.06},
}
GATE_CFG = {
    12: (4500.0, 1200.0),
    24: (5000.0, 1500.0),
    48: (6000.0, 1800.0),
}

def prior_to_vector(prior_dict, names):
    w = np.array([prior_dict.get(n, 0.0) for n in names], dtype=float)
    w = np.clip(w, 0, None)
    if w.sum() <= 0:
        w = np.ones(len(names), dtype=float)
    return w / w.sum()

def assemble_available(h, zone, split_kind, prior_dict):
    store = base_oof if split_kind == "oof" else base_test
    avail = {}
    for key in ["gbsa", "cox", "rsf", "glob_lgb", "glob_cat", "zone_lgb", "lin", "anchor", "timing"]:
        if key == "zone_lgb":
            src = near_lgb_store[split_kind] if zone == "near" else far_lgb_store[split_kind]
            if h in src:
                avail[key] = src[h]
        else:
            if key in store and h in store[key]:
                avail[key] = store[key][h]
    names = [n for n in prior_dict if n in avail]
    if not names:
        names = list(avail.keys())
    P = np.column_stack([avail[n] for n in names])
    prior = prior_to_vector(prior_dict, names)
    return names, P, prior

def optimize_blend(P, y, prior, reg, shrink):
    if P.shape[1] == 1:
        return prior.copy(), prior.copy()
    P = np.clip(P, 1e-6, 1 - 1e-6)
    y = np.asarray(y, dtype=float)

    def objective(w):
        pred = P @ w
        return np.mean((pred - y) ** 2) + reg * np.sum((w - prior) ** 2)

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    bounds = [(0.0, 1.0)] * P.shape[1]
    try:
        res = minimize(
            objective, prior.copy(), method="SLSQP", bounds=bounds, constraints=constraints,
            options={"maxiter": 500, "ftol": 1e-10}
        )
        if res.success:
            opt = np.clip(res.x, 0, None)
            opt = opt / opt.sum()
        else:
            opt = prior.copy()
    except Exception:
        opt = prior.copy()

    final = shrink * opt + (1.0 - shrink) * prior
    final = np.clip(final, 0, None)
    final = final / final.sum()
    return final, opt

weights_log = {}
oof_final = np.zeros((len(train_df), 4), dtype=float)
test_final = np.zeros((len(test_df), 4), dtype=float)

for col_idx, h in enumerate([12, 24, 48]):
    center, width = GATE_CFG[h]
    gate_train = soft_gate(dist_train, center, width)
    gate_test = soft_gate(dist_test, center, width)

    prior_near = PRIORS[(h, "near")]
    names_n, Pn_oof, prior_vec_n = assemble_available(h, "near", "oof", prior_near)
    _, Pn_test, _ = assemble_available(h, "near", "test", prior_near)
    near_mask = mask_map[h] & (dist_train < 5000)
    if near_mask.sum() >= max(8, len(names_n) + 2):
        w_n, opt_n = optimize_blend(
            Pn_oof[near_mask], y_map[h][near_mask], prior_vec_n,
            BLEND_CFG[(h, "near")]["reg"], BLEND_CFG[(h, "near")]["shrink"]
        )
    else:
        w_n, opt_n = prior_vec_n.copy(), prior_vec_n.copy()
    pred_near_oof = np.clip(Pn_oof @ w_n, 0, 1)
    pred_near_test = np.clip(Pn_test @ w_n, 0, 1)

    prior_far = PRIORS[(h, "far")]
    names_f, Pf_oof, prior_vec_f = assemble_available(h, "far", "oof", prior_far)
    _, Pf_test, _ = assemble_available(h, "far", "test", prior_far)
    far_mask = mask_map[h] & ~(dist_train < 5000)
    if far_mask.sum() >= max(8, len(names_f) + 2):
        w_f, opt_f = optimize_blend(
            Pf_oof[far_mask], y_map[h][far_mask], prior_vec_f,
            BLEND_CFG[(h, "far")]["reg"], BLEND_CFG[(h, "far")]["shrink"]
        )
    else:
        w_f, opt_f = prior_vec_f.copy(), prior_vec_f.copy()
    pred_far_oof = np.clip(Pf_oof @ w_f, 0, 1)
    pred_far_test = np.clip(Pf_test @ w_f, 0, 1)

    oof_final[:, col_idx] = gate_train * pred_near_oof + (1.0 - gate_train) * pred_far_oof
    test_final[:, col_idx] = gate_test * pred_near_test + (1.0 - gate_test) * pred_far_test

    weights_log[f"{h}_near"] = {
        "names": names_n,
        "prior": [float(x) for x in prior_vec_n],
        "opt": [float(x) for x in opt_n],
        "final": [float(x) for x in w_n],
    }
    weights_log[f"{h}_far"] = {
        "names": names_f,
        "prior": [float(x) for x in prior_vec_f],
        "opt": [float(x) for x in opt_f],
        "final": [float(x) for x in w_f],
    }

oof_final[:, 3] = 1.0
test_final[:, 3] = 1.0

oof_final = enforce_monotonicity(oof_final)
test_final = enforce_monotonicity(test_final)

exp_oof_final = oof_final.copy()
exp_test_final = test_final.copy()

if HAS_SKSURV:
    gbsa_core_oof = np.column_stack([
        base_oof["gbsa"][12],
        np.clip(base_oof["gbsa"][24] ** 1.10, 0, 1),
        base_oof["gbsa"][48],
        np.ones(len(train_df), dtype=float),
    ])
    gbsa_core_test = np.column_stack([
        base_test["gbsa"][12],
        np.clip(base_test["gbsa"][24] ** 1.10, 0, 1),
        base_test["gbsa"][48],
        np.ones(len(test_df), dtype=float),
    ])
    hard_near_train = dist_train < 5000
    hard_near_test = dist_test < 5000

    core_oof_final = np.zeros((len(train_df), 4), dtype=float)
    core_test_final = np.zeros((len(test_df), 4), dtype=float)

    near12_oof = near_lgb_oof.get(24, anchor_oof[12])
    near12_test = near_lgb_test.get(24, anchor_test[12])

    core_oof_final[:, 0] = np.where(
        hard_near_train,
        0.76 * gbsa_core_oof[:, 0] + 0.12 * base_oof["cox"][12] + 0.02 * base_oof["rsf"][12] + 0.10 * near12_oof,
        gbsa_core_oof[:, 0]
    )
    core_test_final[:, 0] = np.where(
        hard_near_test,
        0.76 * gbsa_core_test[:, 0] + 0.12 * base_test["cox"][12] + 0.02 * base_test["rsf"][12] + 0.10 * near12_test,
        gbsa_core_test[:, 0]
    )

    core_oof_final[:, 1] = np.where(
        hard_near_train,
        0.82 * gbsa_core_oof[:, 1] + 0.14 * base_oof["cox"][24] + 0.02 * base_oof["rsf"][24] + 0.02 * near_lgb_oof[24],
        0.62 * gbsa_core_oof[:, 1] + 0.25 * base_oof["cox"][24] + 0.06 * base_oof["rsf"][24] + 0.07 * far_lgb_oof[24]
    )
    core_test_final[:, 1] = np.where(
        hard_near_test,
        0.82 * gbsa_core_test[:, 1] + 0.14 * base_test["cox"][24] + 0.02 * base_test["rsf"][24] + 0.02 * near_lgb_test[24],
        0.62 * gbsa_core_test[:, 1] + 0.25 * base_test["cox"][24] + 0.06 * base_test["rsf"][24] + 0.07 * far_lgb_test[24]
    )

    core_oof_final[:, 2] = np.where(
        hard_near_train,
        0.73 * gbsa_core_oof[:, 2] + 0.16 * base_oof["cox"][48] + 0.03 * base_oof["rsf"][48] + 0.08 * near_lgb_oof[48],
        0.35 * gbsa_core_oof[:, 2] + 0.22 * base_oof["cox"][48] + 0.06 * base_oof["rsf"][48] + 0.37 * far_lgb_oof[48]
    )
    core_test_final[:, 2] = np.where(
        hard_near_test,
        0.73 * gbsa_core_test[:, 2] + 0.16 * base_test["cox"][48] + 0.03 * base_test["rsf"][48] + 0.08 * near_lgb_test[48],
        0.35 * gbsa_core_test[:, 2] + 0.22 * base_test["cox"][48] + 0.06 * base_test["rsf"][48] + 0.37 * far_lgb_test[48]
    )

    core_oof_final[:, 3] = 1.0
    core_test_final[:, 3] = 1.0
    core_oof_final = enforce_monotonicity(np.clip(core_oof_final, 0, 1))
    core_test_final = enforce_monotonicity(np.clip(core_test_final, 0, 1))
else:
    core_oof_final = exp_oof_final.copy()
    core_test_final = exp_test_final.copy()

weights_log["meta_core_mix"] = {"core_weight": 0.70, "exploratory_weight": 0.30}
oof_final = enforce_monotonicity(np.clip(0.70 * core_oof_final + 0.30 * exp_oof_final, 0, 1))
test_final = enforce_monotonicity(np.clip(0.70 * core_test_final + 0.30 * exp_test_final, 0, 1))
oof_final[:, 3] = 1.0
test_final[:, 3] = 1.0

if DO_OOF:
    exp_hybrid, exp_c_idx, exp_wb, exp_b24, exp_b48, exp_b72 = compute_hybrid_score(
        time_values, event_values, exp_oof_final[:, 1], exp_oof_final[:, 2], exp_oof_final[:, 3]
    )
    core_hybrid, core_c_idx, core_wb, core_b24, core_b48, core_b72 = compute_hybrid_score(
        time_values, event_values, core_oof_final[:, 1], core_oof_final[:, 2], core_oof_final[:, 3]
    )
    print(f"OOF Exploratory Hybrid = {exp_hybrid:.6f} | C={exp_c_idx:.6f} | WB={exp_wb:.6f}")
    print(f"OOF Core        Hybrid = {core_hybrid:.6f} | C={core_c_idx:.6f} | WB={core_wb:.6f}")

if DO_OOF:
    hybrid, c_idx, wb, b24, b48, b72 = compute_hybrid_score(
        time_values, event_values, oof_final[:, 1], oof_final[:, 2], oof_final[:, 3]
    )
    b12 = compute_brier(time_values, event_values, oof_final[:, 0], 12)
    print(f"OOF Hybrid = {hybrid:.6f}")
    print(f"OOF C-Index = {c_idx:.6f}")
    print(f"OOF WBrier = {wb:.6f}")
    print(f"OOF Brier@12 = {b12:.6f}")
    print(f"OOF Brier@24 = {b24:.6f}")
    print(f"OOF Brier@48 = {b48:.6f}")
    print(f"OOF Brier@72 = {b72:.6f}")

submission = pd.DataFrame({
    "event_id": test_df["event_id"].values,
    "prob_12h": test_final[:, 0],
    "prob_24h": test_final[:, 1],
    "prob_48h": test_final[:, 2],
    "prob_72h": test_final[:, 3],
})
submission = sample_sub[["event_id"]].merge(submission, on="event_id", how="left", validate="one_to_one")

def validate_submission(sub, sample):
    required = ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
    assert list(sub.columns) == required
    assert len(sub) == len(sample)
    assert sub["event_id"].equals(sample["event_id"])
    assert sub["event_id"].nunique() == len(sub)
    vals = sub[required[1:]].values
    assert np.isfinite(vals).all()
    assert ((vals >= 0) & (vals <= 1)).all()
    assert np.all(vals[:, 0] <= vals[:, 1] + 1e-12)
    assert np.all(vals[:, 1] <= vals[:, 2] + 1e-12)
    assert np.all(vals[:, 2] <= vals[:, 3] + 1e-12)

validate_submission(submission, sample_sub)

os.makedirs(WORK_DIR, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)

if DO_OOF:
    oof_dump = pd.DataFrame({
        "event_id": train_df["event_id"].values,
        "prob_12h": oof_final[:, 0],
        "prob_24h": oof_final[:, 1],
        "prob_48h": oof_final[:, 2],
        "prob_72h": oof_final[:, 3],
        "event": event_values,
        "time_to_hit_hours": time_values,
    })
    oof_dump.to_csv(os.path.join(WORK_DIR, "oof_predictions.csv"), index=False)

with open(os.path.join(WORK_DIR, "blend_weights.json"), "w") as f:
    json.dump(weights_log, f, indent=2)

print("Saved:", OUTPUT_PATH)
print(submission.head())



# ============================
# v2 post-stack enhancement
# ============================
import glob, re, json, os
SEARCH_EXTERNAL_SUBMISSIONS = True
SAFE_INTERNAL_PATH = os.path.join(WORK_DIR, "submission_internal_safe.csv")
AGG_INTERNAL_PATH = os.path.join(WORK_DIR, "submission_internal_aggressive.csv")
WEIGHTS_PATH = os.path.join(WORK_DIR, "blend_weights_v2.json")

EXP_SEARCH_BLEND = 0.30
SPLIT_GRID = {
    12: [4500.0, 5000.0, 5500.0],
    24: [4500.0, 5000.0, 5500.0],
    48: [5000.0, 5500.0, 6000.0],
}
GATE_SEARCH = {
    12: [(4200.0, 1000.0), (4500.0, 1200.0), (5000.0, 1400.0)],
    24: [(4500.0, 1200.0), (5000.0, 1500.0), (5500.0, 1800.0)],
    48: [(5500.0, 1500.0), (6000.0, 1800.0), (6500.0, 2200.0)],
}

def build_exploratory_for_h(h, split_thr, gate_center, gate_width):
    gate_train = soft_gate(dist_train, gate_center, gate_width)
    gate_test = soft_gate(dist_test, gate_center, gate_width)

    prior_near = PRIORS[(h, "near")]
    names_n, Pn_oof, prior_vec_n = assemble_available(h, "near", "oof", prior_near)
    _, Pn_test, _ = assemble_available(h, "near", "test", prior_near)
    near_mask = mask_map[h] & (dist_train < split_thr)
    if near_mask.sum() >= max(8, len(names_n) + 2):
        w_n, _ = optimize_blend(
            Pn_oof[near_mask],
            y_map[h][near_mask],
            prior_vec_n,
            BLEND_CFG[(h, "near")]["reg"],
            BLEND_CFG[(h, "near")]["shrink"],
        )
    else:
        w_n = prior_vec_n.copy()
    pred_near_oof = np.clip(Pn_oof @ w_n, 0, 1)
    pred_near_test = np.clip(Pn_test @ w_n, 0, 1)

    prior_far = PRIORS[(h, "far")]
    names_f, Pf_oof, prior_vec_f = assemble_available(h, "far", "oof", prior_far)
    _, Pf_test, _ = assemble_available(h, "far", "test", prior_far)
    far_mask = mask_map[h] & ~(dist_train < split_thr)
    if far_mask.sum() >= max(8, len(names_f) + 2):
        w_f, _ = optimize_blend(
            Pf_oof[far_mask],
            y_map[h][far_mask],
            prior_vec_f,
            BLEND_CFG[(h, "far")]["reg"],
            BLEND_CFG[(h, "far")]["shrink"],
        )
    else:
        w_f = prior_vec_f.copy()
    pred_far_oof = np.clip(Pf_oof @ w_f, 0, 1)
    pred_far_test = np.clip(Pf_test @ w_f, 0, 1)

    pred_oof = np.clip(gate_train * pred_near_oof + (1.0 - gate_train) * pred_far_oof, 0, 1)
    pred_test = np.clip(gate_test * pred_near_test + (1.0 - gate_test) * pred_far_test, 0, 1)

    info = {
        "split_thr": float(split_thr),
        "gate_center": float(gate_center),
        "gate_width": float(gate_width),
        "near_names": names_n,
        "near_weights": [float(x) for x in w_n],
        "far_names": names_f,
        "far_weights": [float(x) for x in w_f],
        "brier": float(compute_brier(time_values, event_values, pred_oof, h)),
    }
    return pred_oof, pred_test, info

exp_default_oof = exp_oof_final.copy()
exp_default_test = exp_test_final.copy()
exp_v2_oof = exp_default_oof.copy()
exp_v2_test = exp_default_test.copy()

v2_logs = {}
for col_idx, h in enumerate([12, 24, 48]):
    best_oof = exp_default_oof[:, col_idx].copy()
    best_test = exp_default_test[:, col_idx].copy()
    best_loss = compute_brier(time_values, event_values, best_oof, h)
    best_info = {
        "split_thr": 5000.0,
        "gate_center": float(GATE_CFG[h][0]),
        "gate_width": float(GATE_CFG[h][1]),
        "brier": float(best_loss),
    }

    for split_thr in SPLIT_GRID[h]:
        for center, width in GATE_SEARCH[h]:
            cand_oof, cand_test, cand_info = build_exploratory_for_h(h, split_thr, center, width)
            if cand_info["brier"] < best_loss - 1e-12:
                best_loss = cand_info["brier"]
                best_oof = cand_oof
                best_test = cand_test
                best_info = cand_info

    exp_v2_oof[:, col_idx] = np.clip(
        (1.0 - EXP_SEARCH_BLEND) * exp_default_oof[:, col_idx] + EXP_SEARCH_BLEND * best_oof,
        0,
        1,
    )
    exp_v2_test[:, col_idx] = np.clip(
        (1.0 - EXP_SEARCH_BLEND) * exp_default_test[:, col_idx] + EXP_SEARCH_BLEND * best_test,
        0,
        1,
    )
    v2_logs[f"exploratory_h{h}"] = {
        "default_brier": float(compute_brier(time_values, event_values, exp_default_oof[:, col_idx], h)),
        "best": best_info,
        "blend_weight_to_best": EXP_SEARCH_BLEND,
    }

exp_v2_oof[:, 3] = 1.0
exp_v2_test[:, 3] = 1.0
exp_v2_oof = enforce_monotonicity(exp_v2_oof)
exp_v2_test = enforce_monotonicity(exp_v2_test)

if DO_OOF:
    exp_v2_hybrid, exp_v2_c, exp_v2_wb, _, _, _ = compute_hybrid_score(
        time_values,
        event_values,
        exp_v2_oof[:, 1],
        exp_v2_oof[:, 2],
        exp_v2_oof[:, 3],
    )
    print(f"OOF Exploratory v2 Hybrid = {exp_v2_hybrid:.6f} | C={exp_v2_c:.6f} | WB={exp_v2_wb:.6f}")

def _get_store_pred(store, key, h, default):
    if key in store and h in store[key]:
        return np.asarray(store[key][h], dtype=float)
    return np.asarray(default, dtype=float)

def _avg_rank_signal(*arrays):
    arrs = []
    for arr in arrays:
        if arr is None:
            continue
        vals = np.asarray(arr, dtype=float)
        if vals.ndim != 1:
            continue
        arrs.append(pd.Series(vals).rank(method="average", pct=True).values)
    if not arrs:
        return np.full(len(train_df), 0.5, dtype=float)
    return np.mean(np.column_stack(arrs), axis=1)

aux24_oof = _avg_rank_signal(
    _get_store_pred(base_oof, "gbsa", 24, anchor_oof[24]),
    _get_store_pred(base_oof, "cox", 24, anchor_oof[24]),
    near_lgb_oof.get(24, anchor_oof[24]),
    global_lgb_oof[24],
    global_cat_oof[24],
    1.0 - train_proc["dist_rank_global"].values,
    train_proc["threat_rank_global"].values,
    train_proc["near_speed_rank"].values,
)
aux48_oof = _avg_rank_signal(
    _get_store_pred(base_oof, "gbsa", 48, anchor_oof[48]),
    _get_store_pred(base_oof, "cox", 48, anchor_oof[48]),
    far_lgb_oof.get(48, anchor_oof[48]),
    near_lgb_oof.get(48, anchor_oof[48]),
    global_lgb_oof[48],
    global_cat_oof[48],
    1.0 - train_proc["dist_rank_global"].values,
    train_proc["far_threat_rank"].values,
)
aux24_test = _avg_rank_signal(
    _get_store_pred(base_test, "gbsa", 24, anchor_test[24]),
    _get_store_pred(base_test, "cox", 24, anchor_test[24]),
    near_lgb_test.get(24, anchor_test[24]),
    global_lgb_test[24],
    global_cat_test[24],
    1.0 - test_proc["dist_rank_global"].values,
    test_proc["threat_rank_global"].values,
    test_proc["near_speed_rank"].values,
)
aux48_test = _avg_rank_signal(
    _get_store_pred(base_test, "gbsa", 48, anchor_test[48]),
    _get_store_pred(base_test, "cox", 48, anchor_test[48]),
    far_lgb_test.get(48, anchor_test[48]),
    near_lgb_test.get(48, anchor_test[48]),
    global_lgb_test[48],
    global_cat_test[48],
    1.0 - test_proc["dist_rank_global"].values,
    test_proc["far_threat_rank"].values,
)

_comp_i = []
_comp_j = []
for _i in range(len(time_values)):
    if event_values[_i] != 1:
        continue
    for _j in range(len(time_values)):
        if _i == _j or time_values[_i] >= time_values[_j]:
            continue
        _comp_i.append(_i)
        _comp_j.append(_j)
_comp_i = np.asarray(_comp_i, dtype=int)
_comp_j = np.asarray(_comp_j, dtype=int)

_valid24 = ~((event_values == 0) & (time_values < 24))
_valid48 = ~((event_values == 0) & (time_values < 48))
_y24 = ((event_values == 1) & (time_values <= 24)).astype(float)[_valid24]
_y48 = ((event_values == 1) & (time_values <= 48)).astype(float)[_valid48]

def fast_hybrid_24_48(p24, p48):
    p24 = np.asarray(p24, dtype=float)
    p48 = np.asarray(p48, dtype=float)
    risk = 0.3 * p24 + 0.4 * p48 + 0.3
    ri = risk[_comp_i]
    rj = risk[_comp_j]
    c_idx = ((ri > rj).sum() + 0.5 * (ri == rj).sum()) / len(_comp_i)
    b24 = np.mean((np.clip(p24[_valid24], 0, 1) - _y24) ** 2)
    b48 = np.mean((np.clip(p48[_valid48], 0, 1) - _y48) ** 2)
    wb = 0.3 * b24 + 0.4 * b48
    return 0.3 * c_idx + 0.7 * (1.0 - wb)

def apply_tilt(prob, aux_rank, strength):
    prob = np.asarray(prob, dtype=float)
    aux_rank = np.asarray(aux_rank, dtype=float)
    return np.clip(prob + strength * (aux_rank - 0.5) * prob * (1.0 - prob), 0, 1)

def build_meta_preds(core_arr, exp_arr, aux24, aux48, params):
    w24 = float(params.get("w24", 0.70))
    w48 = float(params.get("w48", 0.70))
    w12 = float(params.get("w12", min(0.85, 0.05 + 0.5 * (w24 + w48))))
    pow24 = float(params.get("pow24", 1.00))
    pow48 = float(params.get("pow48", 1.00))
    tilt24 = float(params.get("tilt24", 0.0))
    tilt48 = float(params.get("tilt48", 0.0))

    out = np.zeros_like(core_arr, dtype=float)
    out[:, 0] = np.clip(w12 * core_arr[:, 0] + (1.0 - w12) * exp_arr[:, 0], 0, 1)
    out[:, 1] = np.clip(w24 * core_arr[:, 1] + (1.0 - w24) * exp_arr[:, 1], 0, 1)
    out[:, 2] = np.clip(w48 * core_arr[:, 2] + (1.0 - w48) * exp_arr[:, 2], 0, 1)

    if abs(pow24 - 1.0) > 1e-12:
        out[:, 1] = np.clip(out[:, 1] ** pow24, 0, 1)
    if abs(pow48 - 1.0) > 1e-12:
        out[:, 2] = np.clip(out[:, 2] ** pow48, 0, 1)
    if abs(tilt24) > 1e-12:
        out[:, 1] = apply_tilt(out[:, 1], aux24, tilt24)
    if abs(tilt48) > 1e-12:
        out[:, 2] = apply_tilt(out[:, 2], aux48, tilt48)

    out[:, 3] = 1.0
    out = enforce_monotonicity(out)
    out[:, 3] = 1.0
    return out

meta_default_params = {
    "w24": 0.70,
    "w48": 0.70,
    "w12": 0.75,
    "pow24": 1.00,
    "pow48": 1.00,
    "tilt24": 0.0,
    "tilt48": 0.0,
}
oof_meta_default = build_meta_preds(core_oof_final, exp_v2_oof, aux24_oof, aux48_oof, meta_default_params)
meta_default_score = fast_hybrid_24_48(oof_meta_default[:, 1], oof_meta_default[:, 2])

meta_best_params = dict(meta_default_params)
meta_best_score = meta_default_score

for w24 in [0.35, 0.45, 0.55, 0.65, 0.75]:
    for w48 in [0.35, 0.45, 0.55, 0.65, 0.75]:
        w12 = min(0.85, 0.05 + 0.5 * (w24 + w48))
        for pow24 in [1.00, 1.04, 1.08, 1.12]:
            for pow48 in [1.00, 1.02, 1.04]:
                for tilt24 in [0.0, 0.01, 0.02]:
                    for tilt48 in [0.0, 0.01, 0.02, 0.03]:
                        cand_params = {
                            "w24": w24,
                            "w48": w48,
                            "w12": w12,
                            "pow24": pow24,
                            "pow48": pow48,
                            "tilt24": tilt24,
                            "tilt48": tilt48,
                        }
                        cand_oof = build_meta_preds(core_oof_final, exp_v2_oof, aux24_oof, aux48_oof, cand_params)
                        cand_score = fast_hybrid_24_48(cand_oof[:, 1], cand_oof[:, 2])
                        if cand_score > meta_best_score + 1e-12:
                            meta_best_score = cand_score
                            meta_best_params = cand_params

meta_safe_params = {
    "w24": 0.45 * 0.70 + 0.55 * meta_best_params["w24"],
    "w48": 0.45 * 0.70 + 0.55 * meta_best_params["w48"],
    "w12": 0.45 * 0.75 + 0.55 * meta_best_params["w12"],
    "pow24": 0.35 * 1.00 + 0.65 * meta_best_params["pow24"],
    "pow48": 0.35 * 1.00 + 0.65 * meta_best_params["pow48"],
    "tilt24": 0.80 * meta_best_params["tilt24"],
    "tilt48": 0.80 * meta_best_params["tilt48"],
}

oof_internal_safe = build_meta_preds(core_oof_final, exp_v2_oof, aux24_oof, aux48_oof, meta_safe_params)
test_internal_safe = build_meta_preds(core_test_final, exp_v2_test, aux24_test, aux48_test, meta_safe_params)

oof_internal_aggressive = build_meta_preds(core_oof_final, exp_v2_oof, aux24_oof, aux48_oof, meta_best_params)
test_internal_aggressive = build_meta_preds(core_test_final, exp_v2_test, aux24_test, aux48_test, meta_best_params)

if DO_OOF:
    safe_hybrid, safe_c, safe_wb, _, _, _ = compute_hybrid_score(
        time_values, event_values, oof_internal_safe[:, 1], oof_internal_safe[:, 2], oof_internal_safe[:, 3]
    )
    aggr_hybrid, aggr_c, aggr_wb, _, _, _ = compute_hybrid_score(
        time_values, event_values, oof_internal_aggressive[:, 1], oof_internal_aggressive[:, 2], oof_internal_aggressive[:, 3]
    )
    print(f"OOF Internal Safe       = {safe_hybrid:.6f} | C={safe_c:.6f} | WB={safe_wb:.6f}")
    print(f"OOF Internal Aggressive = {aggr_hybrid:.6f} | C={aggr_c:.6f} | WB={aggr_wb:.6f}")

weights_log["post_stack_v2"] = {
    "exploratory_search": v2_logs,
    "meta_default": meta_default_params,
    "meta_best": meta_best_params,
    "meta_safe": meta_safe_params,
}

KNOWN_PUBLIC_WEIGHTS = {
    "0.97055.csv": 0.08,
    "0.97092.csv": 0.08,
    "0.97124.csv": 0.08,
    "0.97167.csv": 0.68,
    "0.97169.csv": 0.08,
}

IGNORE_BASENAMES = {
    "train.csv",
    "test.csv",
    "sample_submission.csv",
    "metaData.csv",
    "metadata.csv",
    os.path.basename(OUTPUT_PATH),
    os.path.basename(SAFE_INTERNAL_PATH),
    os.path.basename(AGG_INTERNAL_PATH),
    "oof_predictions.csv",
    "oof_predictions_safe.csv",
    "oof_predictions_aggressive.csv",
}

def align_submission_like(df, sample):
    required = ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
    if not set(required).issubset(df.columns):
        return None
    temp = df[required].copy()
    if temp["event_id"].duplicated().any():
        return None
    if set(temp["event_id"]) != set(sample["event_id"]):
        return None
    merged = sample[["event_id"]].merge(temp, on="event_id", how="left", validate="one_to_one")
    if merged.isna().any().any():
        return None
    vals = merged[required[1:]].values.astype(float)
    if not np.isfinite(vals).all():
        return None
    vals = np.clip(vals, 0, 1)
    vals = enforce_monotonicity(vals)
    vals[:, 3] = 1.0
    merged[required[1:]] = vals
    return merged

def collect_external_candidates(sample):
    if not SEARCH_EXTERNAL_SUBMISSIONS:
        return []
    roots = ["/kaggle/input", "/kaggle/working", ".", "/mnt/data"]
    paths = []
    for root in roots:
        if os.path.isdir(root):
            paths.extend(glob.glob(os.path.join(root, "**", "*.csv"), recursive=True))
    found = []
    seen = set()
    for path in sorted(set(paths)):
        base = os.path.basename(path)
        low = base.lower()
        if base in IGNORE_BASENAMES or path in seen:
            continue
        try:
            if os.path.getsize(path) > 2_500_000:
                continue
        except Exception:
            continue
        interesting = (
            base in KNOWN_PUBLIC_WEIGHTS
            or "submission" in low
            or "blend" in low
            or "public" in low
            or "bayes" in low
            or "0.96" in low
            or "0.97" in low
            or "0.98" in low
            or "0.99" in low
        )
        if not interesting:
            continue
        try:
            df = pd.read_csv(path)
        except Exception:
            continue
        aligned = align_submission_like(df, sample)
        if aligned is None:
            continue
        found.append({"path": path, "base": base, "df": aligned})
        seen.add(path)
    return found

def build_external_blend(candidates):
    if not candidates:
        return None, {"count": 0, "chosen": [], "mode": "none"}
    known = [c for c in candidates if c["base"] in KNOWN_PUBLIC_WEIGHTS]
    if len(known) >= 3:
        chosen = known
        weights = np.array([KNOWN_PUBLIC_WEIGHTS[c["base"]] for c in chosen], dtype=float)
        weights = weights / weights.sum()
        mode = "known_public"
    else:
        scored = []
        for c in candidates:
            m = re.search(r"(0\.\d{3,5})", c["base"])
            score = float(m.group(1)) if m else 0.9700
            scored.append((score, c))
        scored = sorted(scored, key=lambda x: (x[0], x[1]["base"]), reverse=True)[:8]
        chosen = [c for _, c in scored]
        raw_w = np.array([max(score - 0.95, 0.01) for score, _ in scored], dtype=float)
        weights = raw_w / raw_w.sum()
        mode = "generic"

    arr = np.stack(
        [c["df"][["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].values for c in chosen],
        axis=2,
    )
    prob_blend = np.tensordot(arr, weights, axes=([2], [0]))
    rank_blend = np.zeros_like(prob_blend)
    for j in range(prob_blend.shape[1]):
        rank_mat = np.column_stack(
            [pd.Series(arr[:, j, k]).rank(method="average", pct=True).values for k in range(arr.shape[2])]
        )
        rank_blend[:, j] = rank_mat @ weights

    ext = np.clip(0.75 * prob_blend + 0.25 * rank_blend, 0, 1)
    ext[:, 3] = 1.0
    ext = enforce_monotonicity(ext)

    info = {
        "count": len(candidates),
        "chosen": [c["base"] for c in chosen],
        "weights": [float(x) for x in weights],
        "mode": mode,
    }
    return ext, info

external_candidates = collect_external_candidates(sample_sub)
external_blend, external_info = build_external_blend(external_candidates)

test_final_v2 = test_internal_safe.copy()
if external_blend is not None:
    known_count = sum(name in KNOWN_PUBLIC_WEIGHTS for name in external_info["chosen"])
    ext_w = 0.08 if known_count >= 3 else 0.05
    test_final_v2 = enforce_monotonicity(
        np.clip((1.0 - ext_w) * test_internal_safe + ext_w * external_blend, 0, 1)
    )
    test_final_v2[:, 3] = 1.0
    external_info["final_weight"] = float(ext_w)
    print(f"External blend detected: {external_info['mode']} | files={len(external_info['chosen'])} | weight={ext_w:.2f}")
else:
    print("External blend detected: none")

weights_log["external_candidates_v2"] = external_info

def make_submission_from_preds(preds):
    preds = np.asarray(preds, dtype=float)
    preds = enforce_monotonicity(np.clip(preds, 0, 1))
    preds[:, 3] = 1.0
    sub = pd.DataFrame({
        "event_id": test_df["event_id"].values,
        "prob_12h": preds[:, 0],
        "prob_24h": preds[:, 1],
        "prob_48h": preds[:, 2],
        "prob_72h": preds[:, 3],
    })
    sub = sample_sub[["event_id"]].merge(sub, on="event_id", how="left", validate="one_to_one")
    validate_submission(sub, sample_sub)
    return sub

submission_internal_safe = make_submission_from_preds(test_internal_safe)
submission_internal_aggressive = make_submission_from_preds(test_internal_aggressive)
submission_v2 = make_submission_from_preds(test_final_v2)

submission_internal_safe.to_csv(SAFE_INTERNAL_PATH, index=False)
submission_internal_aggressive.to_csv(AGG_INTERNAL_PATH, index=False)
submission_v2.to_csv(OUTPUT_PATH, index=False)

if DO_OOF:
    pd.DataFrame({
        "event_id": train_df["event_id"].values,
        "prob_12h": oof_internal_safe[:, 0],
        "prob_24h": oof_internal_safe[:, 1],
        "prob_48h": oof_internal_safe[:, 2],
        "prob_72h": oof_internal_safe[:, 3],
        "event": event_values,
        "time_to_hit_hours": time_values,
    }).to_csv(os.path.join(WORK_DIR, "oof_predictions_safe.csv"), index=False)
    pd.DataFrame({
        "event_id": train_df["event_id"].values,
        "prob_12h": oof_internal_aggressive[:, 0],
        "prob_24h": oof_internal_aggressive[:, 1],
        "prob_48h": oof_internal_aggressive[:, 2],
        "prob_72h": oof_internal_aggressive[:, 3],
        "event": event_values,
        "time_to_hit_hours": time_values,
    }).to_csv(os.path.join(WORK_DIR, "oof_predictions_aggressive.csv"), index=False)

with open(WEIGHTS_PATH, "w") as f:
    json.dump(weights_log, f, indent=2)

submission = submission_v2.copy()
print("Saved primary :", OUTPUT_PATH)
print("Saved safe    :", SAFE_INTERNAL_PATH)
print("Saved aggr    :", AGG_INTERNAL_PATH)
print("Saved weights :", WEIGHTS_PATH)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 9.3 MB/s eta 0:00:00
DATA_DIR = /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
TRAIN = (221, 37) TEST = (95, 35) HAS_SKSURV = True
anchor@12  Brier = 0.068845
anchor@24  Brier = 0.029693
anchor@48  Brier = 0.019065
global_lgb@12  Brier = 0.059245
global_cat@12  Brier = 0.055916
linear@12      Brier = 0.057712
global_lgb@24  Brier = 0.027575
global_cat@24  Brier = 0.026686
linear@24      Brier = 0.029833
global_lgb@48  Brier = 0.015347
global_cat@48  Brier = 0.015392
linear@48      Brier = 0.017817
near_lgb@12    Brier = 0.056701
near_lgb@24    Brier = 0.027141
near_lgb@48    Brier = 0.014885
far_lgb@24     Brier = 0.027604
far_lgb@48     Brier = 0.014642
timing@12      Brier = 0.211033
timing@24      Brier = 0.340257
timing@48      Brier = 0.368051
gbsa@12        Brie